# AutoPET-III nnU-Net Inference (GPU-only)

Phase 2 of the split-architecture segmentation pipeline. Reads paired CT+PET NIfTIs from Drive (built by `preprocess_autopet_iii.ipynb`), runs nnU-Net inference in chunks, writes SEG + softmax outputs back to Drive.

**Run this on a GPU runtime (L4 recommended).** Total inference time ~3-4 hours on L4 for 597 cases.

**Resume invariant:** a case is considered done if `{pt_uid}.nii.gz` exists in `segmentations/`. Re-running this notebook skips done cases. Cold-start cost is paid once per chunk (vs once per batch in the original architecture).

**Pre-registration:** https://doi.org/10.17605/OSF.IO/4KAZN (§5.2.2)
**Checkpoint:** Zenodo [14007247](https://zenodo.org/records/14007247) (LesionTracer, MD5 `566016409b0bd14770c0b57c1f2873f1`)

In [ ]:
# Working-directory configuration:
# Set the WORK_DIR environment variable (or override here) to point at
# the local or networked folder that holds the raw cohort data.
# Default Colab convention: the mounted Drive root.
import os
WORK_DIR = os.environ.get('WORK_DIR', '/content/drive/MyDrive')
print(f'WORK_DIR = {WORK_DIR}')


In [ ]:
# Step 1: Mount Drive, check GPU
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, time, json
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
print(f'GPU: {gpu.stdout.strip() if gpu.returncode==0 else "NO GPU"}')

DRIVE_ROOT = f'{WORK_DIR}/autopet_iii'
PAIRED_DIR = f'{DRIVE_ROOT}/paired_inputs'
SEG_DIR = f'{DRIVE_ROOT}/segmentations'
os.makedirs(SEG_DIR, exist_ok=True)

n_pairs = sum(1 for f in os.listdir(PAIRED_DIR) if f.endswith('_0001.nii.gz')) if os.path.isdir(PAIRED_DIR) else 0
n_seg = sum(1 for f in os.listdir(SEG_DIR) if f.endswith('.nii.gz'))
print(f'Paired NIfTIs on Drive: {n_pairs}')
print(f'SEGs on Drive:          {n_seg}')
print(f'To infer:               {n_pairs - n_seg}')

In [ ]:
# Step 2: Install LesionTracer fork (NOT vanilla nnunetv2)
REPO_DIR = '/content/autopet-3-submission'
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/MIC-DKFZ/autopet-3-submission.git {REPO_DIR}
!pip install -q -e {REPO_DIR}
import torch
print(f'torch {torch.__version__}; CUDA: {torch.cuda.is_available()}')

In [ ]:
# Step 3: Download/cache LesionTracer checkpoint, verify MD5, extract
import hashlib, urllib.request, zipfile

ZENODO_URL = 'https://zenodo.org/records/14007247/files/autoPET-3-LesionTracer.zip'
EXPECTED_MD5 = '566016409b0bd14770c0b57c1f2873f1'
DRIVE_ZIP = f'{DRIVE_ROOT}/checkpoints/autoPET-3-LesionTracer.zip'
LOCAL_DIR = '/content/lesiontracer'
os.makedirs(os.path.dirname(DRIVE_ZIP), exist_ok=True)
os.makedirs(LOCAL_DIR, exist_ok=True)

def md5_of(path, chunk=4*1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        while (b := f.read(chunk)): h.update(b)
    return h.hexdigest()

if not os.path.exists(DRIVE_ZIP) or md5_of(DRIVE_ZIP) != EXPECTED_MD5:
    print('Downloading 3.8 GB checkpoint...')
    urllib.request.urlretrieve(ZENODO_URL, DRIVE_ZIP)
    assert md5_of(DRIVE_ZIP) == EXPECTED_MD5, 'MD5 mismatch'
print(f'Checkpoint MD5 verified')

if not os.listdir(LOCAL_DIR):
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(LOCAL_DIR)
print(f'Extracted to {LOCAL_DIR}')

In [ ]:
# Step 4: Discover MODEL_FOLDER
import glob
plans_paths = glob.glob(f'{LOCAL_DIR}/**/plans.json', recursive=True)
MODEL_FOLDER = os.path.dirname(plans_paths[0])
print(f'MODEL_FOLDER = {MODEL_FOLDER}')
folds = sorted([d for d in os.listdir(MODEL_FOLDER) if d.startswith('fold_')])
print(f'Folds present: {folds}')

In [ ]:
# Step 4b: PyTorch 2.6 compat shim for legacy nnU-Net checkpoints
target = '/content/autopet-3-submission/nnunetv2/inference/predict_from_raw_data.py'
shim_marker = '# === PyTorch 2.6 compat shim for legacy checkpoints ==='
shim = f'''
{shim_marker}
import torch as _torch_compat
_orig_torch_load = _torch_compat.load
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(*args, **kwargs)
_torch_compat.load = _patched_torch_load
# === end shim ===
'''
with open(target) as f: src = f.read()
if shim_marker not in src:
    with open(target, 'w') as f:
        f.write(src.replace('import torch\n', 'import torch\n' + shim, 1))
    print('Patched torch.load shim')
else:
    print('Shim already present')

# Clear bytecode cache
for root, dirs, files in os.walk('/content/autopet-3-submission'):
    if '__pycache__' in dirs:
        shutil.rmtree(os.path.join(root, '__pycache__'), ignore_errors=True)

In [ ]:
# Step 5: CHUNKED INFERENCE -- run nnU-Net on all paired NIfTIs in chunks
# Strategy: copy CHUNK_SIZE pairs from Drive -> /content/inference_input,
# run nnUNetv2_predict (writes SEG + npz directly to Drive's segmentations/),
# cleanup, repeat. Resume invariant: skip cases whose SEG already exists.

CHUNK_SIZE = 50           # 50 cases × ~200 MB pairs = ~10 GB peak on /content
FOLDS = '0'               # '0' single fold; 'all' for 5-fold ensemble

from datetime import datetime
HEARTBEAT = f'{DRIVE_ROOT}/_inference_heartbeat.json'
ERROR_LOG = f'{DRIVE_ROOT}/_inference_errors.txt'
STAGE_DIR = '/content/inference_input'

def hb(state, **extra):
    payload = {'state': state, 'ts': datetime.now().isoformat(), **extra}
    tmp = HEARTBEAT + '.tmp'
    with open(tmp, 'w') as f: json.dump(payload, f, indent=2)
    os.replace(tmp, HEARTBEAT)

# Determine work: pt_uids that have paired NIfTIs but no SEG yet
all_pt_uids = {f[:-len('_0001.nii.gz')] for f in os.listdir(PAIRED_DIR)
               if f.endswith('_0001.nii.gz')}
existing_seg = {f[:-7] for f in os.listdir(SEG_DIR) if f.endswith('.nii.gz')}
todo_uids = sorted(all_pt_uids - existing_seg)
n_chunks = (len(todo_uids) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f'Pairs available:    {len(all_pt_uids)}')
print(f'Already segmented:  {len(existing_seg)}')
print(f'To infer:           {len(todo_uids)}  ({n_chunks} chunks of {CHUNK_SIZE})')
print(f'Folds:              {FOLDS}\n', flush=True)

session_done = session_failed = 0
session_start = time.time()
hb('session_start', total_todo=len(todo_uids), n_chunks=n_chunks)

for ci in range(n_chunks):
    chunk = todo_uids[ci*CHUNK_SIZE:(ci+1)*CHUNK_SIZE]
    cnum = ci + 1
    print(f'=== Chunk {cnum}/{n_chunks} ({len(chunk)} cases) ===', flush=True)
    hb('chunk_start', chunk=cnum, of=n_chunks, session_done=session_done)

    # Stage chunk to /content (faster nnU-Net I/O than Drive FUSE)
    shutil.rmtree(STAGE_DIR, ignore_errors=True)
    os.makedirs(STAGE_DIR, exist_ok=True)
    t0 = time.time()
    for pt_uid in chunk:
        for suffix in ('_0000.nii.gz', '_0001.nii.gz'):
            src = os.path.join(PAIRED_DIR, f'{pt_uid}{suffix}')
            dst = os.path.join(STAGE_DIR, f'{pt_uid}{suffix}')
            shutil.copy2(src, dst)
    print(f'  staged in {time.time()-t0:.0f}s', flush=True)

    # Inference -- writes SEG + npz + pkl directly to Drive
    hb('inference', chunk=cnum, n_cases=len(chunk))
    t0 = time.time()
    result = subprocess.run([
        'nnUNetv2_predict_from_modelfolder',
        '-i', STAGE_DIR, '-o', SEG_DIR, '-m', MODEL_FOLDER,
        '-f', FOLDS, '--save_probabilities', '--disable_progress_bar',
    ], capture_output=True, text=True)
    el = time.time() - t0
    if result.returncode != 0:
        with open(ERROR_LOG, 'a') as f:
            f.write(f'INFER\t{datetime.now().isoformat()}\tchunk_{cnum}\t'
                    f'{result.stderr[-500:]}\n')
        print(f'  INFERENCE FAILED ({el:.0f}s) -- last stderr: '
              f'{result.stderr[-300:]}', flush=True)
        session_failed += len(chunk)
    else:
        n_seg_new = sum(1 for u in chunk
                        if os.path.exists(os.path.join(SEG_DIR, f'{u}.nii.gz')))
        session_done += n_seg_new
        rate = el / max(n_seg_new, 1)
        cum = (time.time() - session_start) / 60
        print(f'  inferred {n_seg_new}/{len(chunk)} in {el:.0f}s ({rate:.0f}s/case) | '
              f'session: {session_done} done / {session_failed} failed / '
              f'{cum:.1f} min', flush=True)
    hb('chunk_done', chunk=cnum, session_done=session_done,
       session_failed=session_failed)
    print('', flush=True)

shutil.rmtree(STAGE_DIR, ignore_errors=True)
hb('finished', session_done=session_done, session_failed=session_failed,
   total_segs_on_drive=len(os.listdir(SEG_DIR)))
elapsed_hr = (time.time() - session_start) / 3600
print(f'=== Inference done in {elapsed_hr:.2f} hr ===')
print(f'  Segmented this session: {session_done}')
print(f'  Failed this session:    {session_failed}')
print(f'  Total SEGs on Drive:    {len(os.listdir(SEG_DIR))}')

In [ ]:
# Step 6: Verify -- count SEGs + softmax + spot-check shapes
import nibabel as nib
import numpy as np
from scipy import ndimage

seg_files = [f for f in os.listdir(SEG_DIR) if f.endswith('.nii.gz')]
npz_files = [f for f in os.listdir(SEG_DIR) if f.endswith('.npz')]
print(f'SEGs (.nii.gz):     {len(seg_files)}')
print(f'Softmax (.npz):     {len(npz_files)}')
print(f'Pairs available:    {sum(1 for f in os.listdir(PAIRED_DIR) if f.endswith("_0001.nii.gz"))}')

if seg_files:
    print(f'\nSpot-check (5 samples):')
    for f in seg_files[:5]:
        seg = nib.load(os.path.join(SEG_DIR, f)).get_fdata()
        _, n = ndimage.label(seg > 0)
        n_pos = int((seg > 0).sum())
        print(f'  {f}: shape={seg.shape}, positive_voxels={n_pos:,}, components={n}')